In [ ]:
import mercantile

bbox = (34.2, 29.0, 35.6, 32.7)

tiles = list(mercantile.tiles(*bbox, zooms=[12]))
for t in tiles:
    print(t.z, t.x, t.y)

12 2437 1653
12 2437 1654
12 2437 1655
12 2437 1656
12 2437 1657
12 2437 1658
12 2437 1659
12 2437 1660
12 2437 1661
12 2437 1662
12 2437 1663
12 2437 1664
12 2437 1665
12 2437 1666
12 2437 1667
12 2437 1668
12 2437 1669
12 2437 1670
12 2437 1671
12 2437 1672
12 2437 1673
12 2437 1674
12 2437 1675
12 2437 1676
12 2437 1677
12 2437 1678
12 2437 1679
12 2437 1680
12 2437 1681
12 2437 1682
12 2437 1683
12 2437 1684
12 2437 1685
12 2437 1686
12 2437 1687
12 2437 1688
12 2437 1689
12 2437 1690
12 2437 1691
12 2437 1692
12 2437 1693
12 2437 1694
12 2437 1695
12 2437 1696
12 2437 1697
12 2437 1698
12 2437 1699
12 2437 1700
12 2437 1701
12 2437 1702
12 2438 1653
12 2438 1654
12 2438 1655
12 2438 1656
12 2438 1657
12 2438 1658
12 2438 1659
12 2438 1660
12 2438 1661
12 2438 1662
12 2438 1663
12 2438 1664
12 2438 1665
12 2438 1666
12 2438 1667
12 2438 1668
12 2438 1669
12 2438 1670
12 2438 1671
12 2438 1672
12 2438 1673
12 2438 1674
12 2438 1675
12 2438 1676
12 2438 1677
12 2438 1678
12 2438 1679

In [ ]:
#pip install requests mercantile mapbox-vector-tile shapely pyproj

In [ ]:
import csv, time, requests, mercantile, mapbox_vector_tile
from shapely.geometry import Point, Polygon
from mercantile import ul

TILE_URL = "https://tiles.queeringthemap.com/{z}/{x}/{y}.mvt"
ZOOM = 12  # 11–14; higher zoom = more tiles (try 12–13)

west_bank_poly = Polygon([(34.2,31.2),(35.6,31.2),(35.6,32.6),(34.2,32.6)])
gaza_poly = Polygon([(34.17,31.2),(34.6,31.2),(34.6,31.6),(34.17,31.6)])

def tile_to_lonlat(x, y, z, px, py, extent=4096):
    tl = ul(x, y, z)       # top-left lon/lat of the tile
    br = ul(x+1, y+1, z)   # bottom-right lon/lat of the tile
    lon = tl.lng + (px/extent)*(br.lng - tl.lng)
    lat = tl.lat + (py/extent)*(br.lat - tl.lat)
    return lon, lat

def tiles_for_bbox(bbox, zoom):
    lon_min, lat_min, lon_max, lat_max = bbox
    return list(mercantile.tiles(lon_min, lat_min, lon_max, lat_max, [zoom]))

rows = []

tile_set = set()
for bbox in [(34.2,31.2,35.6,32.6), (34.17,31.2,34.6,31.6)]:
    for t in tiles_for_bbox(bbox, ZOOM):
        tile_set.add((t.z, t.x, t.y))

for z, x, y in sorted(tile_set):
    url = TILE_URL.format(z=z, x=x, y=y)
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        continue
    decoded = mapbox_vector_tile.decode(r.content)
    for layer_name, layer in decoded.items():
        for feat in layer["features"]:
            geom = feat["geometry"]
            if geom["type"] != "Point":
                continue
            px, py = geom["coordinates"]
            lon, lat = tile_to_lonlat(x, y, z, px, py, extent=layer.get("extent", 4096))
            pt = Point(lon, lat)
            if not (west_bank_poly.contains(pt) or gaza_poly.contains(pt)):
                continue
            props = feat.get("properties", {})
            rows.append({
                "lon": lon,
                "lat": lat,
                "id": props.get("id") or props.get("_id"),
                "text": props.get("text") or props.get("body") or props.get("content"),
                "lang": props.get("lang") or props.get("language"),
                "ts": props.get("timestamp") or props.get("created_at")
            })
    time.sleep(0.2)

with open("qtm_palestine_comments.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["id","lat","lon","lang","ts","text"])
    w.writeheader()
    for r in rows:
        w.writerow(r)


In [ ]:
import requests, mercantile, mapbox_vector_tile

bbox = (34.2, 29.0, 35.6, 32.7)  # Palestine approx
tiles = list(mercantile.tiles(*bbox, zooms=[12]))

ids = set()
for t in tiles:
    url = f"https://tiles.queeringthemap.com/{t.z}/{t.x}/{t.y}.mvt"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        continue
    tile = mapbox_vector_tile.decode(r.content)
    for layer_name, layer in tile.items():
        for f in layer["features"]:
            props = f.get("properties", {})
            if "id" in props:
                ids.add(str(props["id"]))
print(f"Collected {len(ids)} IDs")


Collected 0 IDs


In [ ]:
import requests, mapbox_vector_tile, json

TEST_URL = "https://tiles.queeringthemap.com/basemap/9/306/207.mvt"  # paste the exact one
r = requests.get(TEST_URL, timeout=30)
r.raise_for_status()
tile = mapbox_vector_tile.decode(r.content)

print("Layers:", list(tile.keys()))
for lname, layer in tile.items():
    print("\nLayer:", lname, "| features:", len(layer["features"]))
    for f in layer["features"][:5]:
        print(json.dumps(f.get("properties", {}), ensure_ascii=False))

Layers: ['boundaries', 'earth', 'landuse', 'natural', 'physical_line', 'physical_point', 'places', 'pois', 'roads', 'transit', 'water']

Layer: boundaries | features: 6
{"pmap:kind_detail": "5", "disputed": true, "pmap:kind": "county", "pmap:min_admin_level": 5}
{"pmap:kind_detail": "2", "disputed": true, "pmap:kind": "country", "pmap:min_admin_level": 2}
{"pmap:kind_detail": "4", "pmap:kind": "region", "pmap:min_admin_level": 4}
{"pmap:kind_detail": "6", "pmap:kind": "county", "pmap:min_admin_level": 6}
{"pmap:kind_detail": "2", "pmap:kind": "country", "pmap:min_admin_level": 2}

Layer: earth | features: 1
{"pmap:kind": "earth"}

Layer: landuse | features: 583
{"landuse": "residential", "pmap:kind": "residential"}
{"landuse": "residential", "pmap:kind": "residential", "place": "suburb"}
{"landuse": "residential", "pmap:kind": "residential", "place": "suburb"}
{"landuse": "residential", "pmap:kind": "residential"}
{"landuse": "residential", "pmap:kind": "residential"}

Layer: natural |

In [ ]:
import requests, csv, time

FEATURE_URL = "https://www.queeringthemap.com/data/moments.json"

WB = (34.2, 31.2, 35.6, 32.6) 
GZ = (34.17, 31.2, 34.6, 31.6) 

def in_bbox(lon, lat, bbox):
    lon_min, lat_min, lon_max, lat_max = bbox
    return (lon_min <= lon <= lon_max) and (lat_min <= lat <= lat_max)

hdrs = {"Referer": "https://www.queeringthemap.com", "Accept": "application/json,*/*"}
idx = requests.get(FEATURE_URL, headers=hdrs, timeout=60).json()

ids = []
for feat in idx.get("features", []):
    fid = feat.get("id")
    geom = feat.get("geometry", {})
    if not fid or geom.get("type") != "Point":
        continue
    lon, lat = geom.get("coordinates", [None, None])
    if lon is None or lat is None:
        continue
    if in_bbox(lon, lat, WB) or in_bbox(lon, lat, GZ):
        ids.append(int(fid))

ids = sorted(set(ids))
print(f"IDs in region: {len(ids)}")

def fetch_moment(mid):
    url = f"https://www.queeringthemap.com/moment/{mid}"
    r = requests.get(url, headers=hdrs, timeout=30)
    if not r.ok:
        return None
    try:
        return r.json()
    except Exception:
        return None

rows = []
for i, mid in enumerate(ids, 1):
    data = fetch_moment(mid)
    if data and data.get("description"):
        rows.append({
            "short_id": str(data.get("short_id") or mid),
            "lon": "",
            "lat": "",
            "description": data["description"].replace("\r\n","\n")
        })
    if i % 50 == 0:
        print(f"{i}/{len(ids)}"); time.sleep(0.25)  # polite pacing

with open("qtm_palestine_posts.csv","w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["short_id","lon","lat","description"])
    w.writeheader(); w.writerows(rows)

print("Saved qtm_palestine_posts.csv")


IDs in region: 350
50/350
100/350
150/350
200/350
250/350
300/350
350/350
Saved qtm_palestine_posts.csv


In [ ]:
import requests, csv, time, unicodedata

FEATURE_URL = "https://www.queeringthemap.com/data/moments.json"

WB = (34.10, 31.00, 35.80, 32.85)   # (lon_min, lat_min, lon_max, lat_max)
GZ = (34.15, 31.18, 34.64, 31.63)

def in_bbox(lon, lat, bbox):
    lo, la, hi, ha = bbox
    return (lo <= lon <= hi) and (la <= lat <= ha)

hdrs = {
    "Referer": "https://www.queeringthemap.com",
    "Accept": "application/json,*/*",
    "User-Agent": "OPT-research/1.0"
}

idx = requests.get(FEATURE_URL, headers=hdrs, timeout=120).json()

items = []
for feat in idx.get("features", []):
    sid = feat.get("id")
    g = feat.get("geometry", {})
    if not sid or g.get("type") != "Point":
        continue
    lon, lat = g.get("coordinates", [None, None])
    if lon is None or lat is None:
        continue
    if in_bbox(lon, lat, WB) or in_bbox(lon, lat, GZ):
        items.append({"short_id": int(sid), "lon": lon, "lat": lat})

seen = set()
ids = []
for it in items:
    if it["short_id"] not in seen:
        seen.add(it["short_id"])
        ids.append(it)

print(f"IDs in OPT region: {len(ids)}")

def fetch_moment(mid):
    r = requests.get(f"https://www.queeringthemap.com/moment/{mid}",
                     headers=hdrs, timeout=30)
    if not r.ok:
        return None
    try:
        return r.json()
    except Exception:
        return None

rows = []
for i, it in enumerate(ids, 1):
    data = fetch_moment(it["short_id"])
    if data and data.get("description"):
        txt = unicodedata.normalize("NFC", data["description"]).replace("\r\n", "\n")
        rows.append({
            "short_id": str(data.get("short_id") or it["short_id"]),
            "lon": it["lon"],
            "lat": it["lat"],
            "description": txt
        })
    if i % 50 == 0:
        print(f"{i}/{len(ids)}")
        time.sleep(0.3)

with open("qtm_OPT_posts.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=["short_id","lon","lat","description"],
                       quoting=csv.QUOTE_ALL)
    w.writeheader(); w.writerows(rows)

with open("qtm_OPT_posts.tsv", "w", newline="", encoding="utf-8") as f:
    for r in rows:
        line = f'{r["short_id"]}\t{r["lon"]}\t{r["lat"]}\t{r["description"].replace("\t","  ")}\n'
        f.write(line)

print("Saved qtm_OPT_posts.csv and qtm_OPT_posts.tsv")


IDs in OPT region: 409
50/409
100/409
150/409
200/409
250/409
300/409
350/409
400/409
Saved qtm_OPT_posts.csv and qtm_OPT_posts.tsv
